In [ ]:
#| default_exp therapy_config

# Runtime configuration for string therapy (local dev, pixi , packaged install) 


Host / storage
* ST_PROJECT_ROOT   Default directory for relative paths (default: cwd).
* ST_ENV_FILE       Path to ``.env``. If unset, tries project ``.env`` then cwd.
* ST_DATA_DIR       App data dir; SQLite defaults to ``<dir>/app.sqlite``.
* ST_SQLITE_PATH    Full path to SQLite (overrides data dir).

Dock
* ST_ICON_IDS       Comma-separated, e.g. ``1,2,3,4``. Default: ``1,2,3,4``.

Jupyter (REST ``/api/contents`` for chat → notebook; run ``jupyter lab`` yourself in a terminal)
* JUPYTER_BASE_URL, JUPYTER_HOST, JUPYTER_PORT, JUPYTER_TOKEN, JUPYTER_NOTEBOOK_PATH
* ST_JUPYTER_LAB_URL   Full URL for the Lab iframe (default ``http://127.0.0.1:8888/lab``). Must be reachable **from the web browser** (not only the server): if you open the app via SSH / remote host, use that host's IP/DNS + port or an SSH tunnel — ``localhost`` in the iframe means the **browser machine**.

FastHTML
* ST_PORT           UI port (default ``5007``).
* ST_BIND_HOST      Uvicorn bind address (default ``0.0.0.0``). If ``http://localhost:ST_PORT``
  shows ERR_EMPTY_RESPONSE, open ``http://127.0.0.1:ST_PORT/`` instead, or set ``ST_BIND_HOST=::`` (IPv6).
* ST_RELOAD         Uvicorn reload (dev).
* ST_OPEN_BROWSER   Open browser on start.

LLM: ``LISETTE_*``, ``ANTHROPIC_API_KEY``, ``CLAUDE_API_KEY``.

In [ ]:
#| export
from __future__ import annotations

import os
from pathlib import Path


def _env(*names: str, default: str = "") -> str:
    """First non-empty env among ``names``."""
    for n in names:
        v = (os.getenv(n) or "").strip()
        if v:
            return v
    return default


def project_root() -> Path:
    """Anchor for ``pixi.toml`` / ``.env`` search; defaults to process cwd."""
    raw = _env("ST_PROJECT_ROOT")
    if raw:
        return Path(raw).expanduser().resolve()
    return Path.cwd()


def load_environment() -> None:
    """Load ``.env`` without overriding existing ``os.environ``."""
    try:
        from dotenv import load_dotenv
    except ImportError:
        return

    explicit = _env("ST_ENV_FILE")
    if explicit:
        p = Path(explicit).expanduser()
        if p.is_file():
            load_dotenv(p, override=False)
        return

    for candidate in (
        project_root() / ".env",
        project_root().parent / ".env",
        Path.cwd() / ".env",
    ):
        if candidate.is_file():
            load_dotenv(candidate, override=False)
            return


def sqlite_path() -> Path:
    """
    Resolved path to the vault SQLite database.

    When installed as a library, set ``ST_DATA_DIR`` or ``ST_SQLITE_PATH`` so the DB
    is not under ``site-packages``.
    """
    explicit = _env("ST_SQLITE_PATH")
    if explicit:
        return Path(explicit).expanduser().resolve()

    data = _env("ST_DATA_DIR")
    if data:
        return (Path(data).expanduser().resolve() / "app.sqlite")

    return Path(__file__).resolve().parent / "db" / "app.sqlite"


def dock_icon_ids(default: tuple[int, ...] = (1, 2, 3, 4)) -> tuple[int, ...]:
    """Dock button ids from ``ST_ICON_IDS``, else ``default``."""
    raw = _env("ST_ICON_IDS")
    if raw:
        out: list[int] = []
        for part in raw.split(","):
            part = part.strip()
            if part:
                out.append(int(part))
        return tuple(out) if out else default
    return default


def http_port() -> int:
    """FastHTML / uvicorn port."""
    raw = _env("ST_PORT", default="5007")
    return int(raw)


def http_bind_host() -> str:
    """Address uvicorn binds to (``ST_BIND_HOST``, default ``0.0.0.0``).

    ``0.0.0.0`` listens on all IPv4 interfaces. Many browsers resolve ``localhost`` to IPv6
    (``::1``) first; that does **not** hit an IPv4-only listener, which often shows as
    ERR_EMPTY_RESPONSE. Prefer ``web_app_url()`` in the browser, or set ``ST_BIND_HOST=::``
    so ``localhost`` works (Linux; ``127.0.0.1`` may then need the same stack).
    """
    return _env("ST_BIND_HOST", default="0.0.0.0") or "0.0.0.0"


def web_app_url() -> str:
    """Recommended browser URL (IPv4 loopback). Use this if ``http://localhost:…`` fails."""
    return f"http://127.0.0.1:{http_port()}/"


def uvicorn_reload() -> bool:
    """True when ``ST_RELOAD`` is truthy."""
    v = _env("ST_RELOAD")
    return bool(v) and v.lower() in ("1", "true", "yes", "on")


def open_browser() -> bool:
    """Open UI in browser unless disabled."""
    v = _env("ST_OPEN_BROWSER", default="1").lower()
    return v not in ("0", "false", "no", "off")


def jupyter_base_url() -> str:
    """
    HTTP origin for Jupyter Server (Lab iframe + REST ``/api/contents``).

    ``JUPYTER_BASE_URL`` wins when set (no trailing slash). Otherwise
    ``http://{JUPYTER_HOST}:{JUPYTER_PORT}``.

    If Jupyter binds to a *different* port (e.g. it prints ``http://127.0.0.1:52537`` when 8888
    is taken), set ``JUPYTER_BASE_URL`` to that full origin so the iframe and REST calls match.
    Default host is ``127.0.0.1`` to align with opening the app at ``http://127.0.0.1:ST_PORT``.
    """
    base = (os.getenv("JUPYTER_BASE_URL") or "").strip().rstrip("/")
    if base:
        return base
    port = int(os.getenv("JUPYTER_PORT", "8888"))
    host = (os.getenv("JUPYTER_HOST") or "127.0.0.1").strip() or "127.0.0.1"
    return f"http://{host}:{port}"


def jupyter_lab_link_url() -> str:
    """
    URL for the \"Open Jupyter Lab\" link in the visualization panel.

    Set ``ST_JUPYTER_LAB_URL`` to the URL Jupyter prints when you run ``jupyter lab``.
    If unset, returns a placeholder (``http://127.0.0.1:8888/lab``).
    """
    u = _env("ST_JUPYTER_LAB_URL")
    if u:
        return u.rstrip("/")
    return "http://127.0.0.1:8888/lab"
